# Skin Lesion Segmentation

Unsupervised segmentation of dermoscopic images from the Kaggle
[Skin Cancer: Malignant vs. Benign](https://www.kaggle.com/datasets/fanconic/skin-cancer-malignant-vs-benign)
dataset. Each image is clustered with K-Means, the lesion cluster is cleaned with morphological operations, and a
region of interest (ROI) is cropped around it. The ROIs can be used as input to a downstream classifier.

Team project by J Kowshik and K S Vinay Sathvik (B.E. AIML, B.N.M. Institute of Technology).

In [ ]:
import os
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage as ndi
from skimage.color import rgb2gray
from skimage.io import imread, imsave
from skimage.measure import label, regionprops
from skimage.morphology import disk
from skimage.segmentation import find_boundaries
from skimage.transform import resize
from sklearn.cluster import KMeans

## 1. Data

In [ ]:
DATA_DIR = '/kaggle/input/skin-cancer-malignant-vs-benign/data/train'

if not os.path.isdir(DATA_DIR):
    import kagglehub
    root = kagglehub.dataset_download('fanconic/skin-cancer-malignant-vs-benign')
    DATA_DIR = os.path.join(root, 'data', 'train')

mal_paths = sorted(glob(os.path.join(DATA_DIR, 'malignant', '*.jpg')))
ben_paths = sorted(glob(os.path.join(DATA_DIR, 'benign', '*.jpg')))
print(f'{len(mal_paths)} malignant and {len(ben_paths)} benign training images')

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for row, (paths, name) in zip(axes, [(mal_paths, 'Malignant'), (ben_paths, 'Benign')]):
    for ax, path in zip(row, paths[:6]):
        ax.imshow(imread(path))
        ax.set_title(name)
        ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Method

Most images are taken through a dermoscope, so the corners are often dark. Clustering the raw image puts these corners
in the same cluster as the lesion, which is why the field of view is removed first.

1. **Field of view.** Dark regions connected to the image border are treated as the dermoscope vignette and excluded.
2. **Clustering.** K-Means (k = 2) on the grayscale intensities inside the field of view. The darker cluster is the lesion candidate.
3. **Refinement.** Morphological opening and closing, hole filling, and removal of small blobs. If several regions remain,
   the one that is large and close to the image centre is kept.
4. **ROI.** The bounding box of the final mask, with a small margin, is cropped from the RGB image.

In [ ]:
def field_of_view(gray, dark=0.15):
    regions = label(gray < dark)
    edge = np.concatenate([regions[0], regions[-1], regions[:, 0], regions[:, -1]])
    vignette = np.isin(regions, np.unique(edge[edge > 0]))
    return ~vignette


def elbow(gray, mask, max_k=6):
    pixels = gray[mask].reshape(-1, 1)
    inertia = [KMeans(n_clusters=k, n_init=10, random_state=1).fit(pixels).inertia_ for k in range(1, max_k + 1)]
    plt.plot(range(1, max_k + 1), inertia, 'o-')
    plt.xlabel('k')
    plt.ylabel('Within-cluster sum of squares')
    plt.title('Elbow curve')
    plt.show()


def cluster_lesion(gray, fov, k=2):
    km = KMeans(n_clusters=k, n_init=10, random_state=1).fit(gray[fov].reshape(-1, 1))
    darkest = np.argmin(km.cluster_centers_.ravel())
    mask = np.zeros_like(fov)
    mask[fov] = km.labels_ == darkest
    return mask


def refine(mask, radius=3, min_area=64):
    mask = ndi.binary_opening(mask, structure=disk(radius))
    mask = ndi.binary_closing(mask, structure=disk(radius))
    mask = ndi.binary_fill_holes(mask)

    regions = [r for r in regionprops(label(mask)) if r.area >= min_area]
    if not regions:
        return np.zeros_like(mask)

    h, w = mask.shape
    centre = np.array([h / 2, w / 2])

    def score(r):
        offset = np.linalg.norm(np.array(r.centroid) - centre) / np.hypot(h, w)
        return r.area * (1 - offset) ** 2

    best = max(regions, key=score)
    out = np.zeros_like(mask)
    out[tuple(best.coords.T)] = True
    return out


def crop_roi(image, mask, pad=8):
    rows, cols = np.nonzero(mask)
    if rows.size == 0:
        return image
    r0, r1 = max(rows.min() - pad, 0), min(rows.max() + pad + 1, mask.shape[0])
    c0, c1 = max(cols.min() - pad, 0), min(cols.max() + pad + 1, mask.shape[1])
    return image[r0:r1, c0:c1]


def segment(image):
    gray = rgb2gray(image)
    fov = field_of_view(gray)
    raw = cluster_lesion(gray, fov)
    mask = refine(raw)
    return gray, fov, raw, mask

## 3. One image, step by step

In [ ]:
sample = imread(mal_paths[0])
gray, fov, raw, mask = segment(sample)

elbow(gray, fov)

In [ ]:
overlay = sample.copy()
overlay[find_boundaries(mask)] = [0, 255, 0]

panels = [(sample, 'Input'), (fov, 'Field of view'), (raw, 'Darker K-Means cluster'),
          (mask, 'Refined mask'), (overlay, 'Boundary'), (crop_roi(sample, mask), 'ROI')]
fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
for ax, (img, title) in zip(axes, panels):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Results on more images

In [ ]:
def show_grid(paths, title, n=12):
    fig, axes = plt.subplots(2, n // 2, figsize=(18, 6.5))
    for ax, path in zip(axes.ravel(), paths[:n]):
        image = imread(path)
        mask = segment(image)[-1]
        overlay = image.copy()
        overlay[find_boundaries(mask)] = [0, 255, 0]
        ax.imshow(overlay)
        ax.set_title(os.path.basename(path))
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show_grid(mal_paths, 'Malignant')
show_grid(ben_paths, 'Benign')

## 5. ROI export

ROIs are resized to 128 x 128 and written to `rois/<class>/`, ready for a classifier. Images where no lesion
region survives refinement are counted and skipped.

In [ ]:
OUT_DIR = 'rois'
skipped = 0

for paths, name in [(mal_paths, 'malignant'), (ben_paths, 'benign')]:
    os.makedirs(os.path.join(OUT_DIR, name), exist_ok=True)
    for path in paths:
        image = imread(path)
        mask = segment(image)[-1]
        if not mask.any():
            skipped += 1
            continue
        roi = resize(crop_roi(image, mask), (128, 128), anti_aliasing=True)
        imsave(os.path.join(OUT_DIR, name, os.path.basename(path)), (roi * 255).astype(np.uint8), check_contrast=False)

print(f'Skipped {skipped} images with an empty mask')

In [ ]:
areas = {}
for paths, name in [(mal_paths, 'Malignant'), (ben_paths, 'Benign')]:
    areas[name] = [segment(imread(p))[-1].mean() for p in paths[:200]]

plt.hist(areas['Malignant'], bins=30, alpha=0.6, label='Malignant')
plt.hist(areas['Benign'], bins=30, alpha=0.6, label='Benign')
plt.xlabel('Lesion area (fraction of image)')
plt.ylabel('Images')
plt.title('Segmented lesion area, first 200 images per class')
plt.legend()
plt.show()

## 6. Limitations

- There are no ground-truth masks in this dataset, so the segmentation is checked visually rather than with Dice or IoU.
  The ISIC 2016 Task 1 masks would allow a quantitative evaluation.
- Lesions with low contrast against the skin (for example red vascular lesions) and images with dense hair are the main failure cases.
- No classifier is trained here. The exported ROIs are the intended input for one.